In [8]:
#PM10 aggregation
import zipfile
import xml.etree.ElementTree as ET
import os
from pathlib import Path
import shutil
from collections import defaultdict
import numpy as np
from scipy import ndimage
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from shapely.geometry import Polygon
from rasterio.crs import CRS

# Configuration
INPUT_DIR = "./kmz_files_pm10"
OUTPUT_DIR = "./hotspot_rasters_pm10_v2"
NS = '{http://www.opengis.net/kml/2.2}'
STANDARD = 0.000045  # Standard for hazard quotient calculation

def extract_contour_data(kmz_path):
    """Extract contour level data from a KMZ file"""
    contours = []
    
    try:
        extract_dir = "./temp_kml_contour"
        with zipfile.ZipFile(kmz_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        kml_file = None
        for file in os.listdir(extract_dir):
            if file.endswith('.kml'):
                kml_file = os.path.join(extract_dir, file)
                break
        
        if kml_file is None:
            return contours
        
        tree = ET.parse(kml_file)
        root = tree.getroot()
        
        doc = root.find(f'{NS}Document')
        if doc is None:
            return contours
        
        for folder in doc.findall(f'{NS}Folder'):
            folder_name = folder.find(f'{NS}name')
            if folder_name is None or folder_name.text is None:
                continue
            
            fname = folder_name.text.strip()
            
            if any(skip in fname for skip in ['HYSPLIT', 'NOAA', 'EPA', 'Soure']):
                continue
            
            time_str = fname.replace('<pre>', '').replace('</pre>', '').strip()
            
            for placemark in folder.findall(f'{NS}Placemark'):
                pm_name = placemark.find(f'{NS}name')
                if pm_name is None or pm_name.text is None:
                    continue
                
                name_text = pm_name.text.strip()
                
                if 'Contour Level' in name_text:
                    try:
                        parts = name_text.split(':')
                        if len(parts) > 1:
                            conc_str = parts[1].strip().split()[0]
                            concentration = float(conc_str)
                            
                            coords = None
                            
                            multi_geom = placemark.find(f'{NS}MultiGeometry')
                            if multi_geom is not None:
                                polygons = multi_geom.findall(f'{NS}Polygon')
                                if polygons:
                                    for polygon in polygons:
                                        outer_ring = polygon.find(f'{NS}outerBoundaryIs/{NS}LinearRing/{NS}coordinates')
                                        if outer_ring is not None and outer_ring.text:
                                            coords_text = outer_ring.text.strip()
                                            coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                                            if coords:
                                                break
                                
                                if coords is None:
                                    linestrings = multi_geom.findall(f'{NS}LineString')
                                    if linestrings:
                                        for linestring in linestrings:
                                            line_coords = linestring.find(f'{NS}coordinates')
                                            if line_coords is not None and line_coords.text:
                                                coords_text = line_coords.text.strip()
                                                coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                                                if coords:
                                                    break
                            
                            if coords is None:
                                polygon = placemark.find(f'{NS}Polygon')
                                if polygon is not None:
                                    outer_ring = polygon.find(f'{NS}outerBoundaryIs/{NS}LinearRing/{NS}coordinates')
                                    if outer_ring is not None and outer_ring.text:
                                        coords_text = outer_ring.text.strip()
                                        coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                            
                            if coords is None:
                                linestring = placemark.find(f'{NS}LineString')
                                if linestring is not None:
                                    line_coords = linestring.find(f'{NS}coordinates')
                                    if line_coords is not None and line_coords.text:
                                        coords_text = line_coords.text.strip()
                                        coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                            
                            if coords and len(coords) > 0:
                                contours.append({
                                    'time': time_str,
                                    'concentration': concentration,
                                    'coords': coords
                                })
                    except (ValueError, IndexError):
                        pass
        
        shutil.rmtree(extract_dir)
        
    except Exception as e:
        print(f"Error processing {kmz_path}: {e}")
    
    return contours

def create_raster_stack(all_contours):
    """Create raster stack from contours, aggregate by timestamp, then calculate statistics"""
    
    if not all_contours:
        print("No contours found!")
        return
    
    # Determine bounds
    all_coords = [c for contour in all_contours for c in contour['coords']]
    lons = [c[0] for c in all_coords]
    lats = [c[1] for c in all_coords]
    
    minlon, maxlon = min(lons), max(lons)
    minlat, maxlat = min(lats), max(lats)
    
    # Add buffer
    buffer = 0.05
    minlon -= buffer
    maxlon += buffer
    minlat -= buffer
    maxlat += buffer
    
    # Create grid (0.01 degree resolution)
    grid_res = 0.01
    width = int((maxlon - minlon) / grid_res)
    height = int((maxlat - minlat) / grid_res)
    
    print(f"Creating raster grid: {width} x {height}")
    
    # Group contours by timestamp
    contours_by_time = defaultdict(list)
    for contour in all_contours:
        contours_by_time[contour['time']].append(contour)
    
    print(f"Found {len(contours_by_time)} unique timestamps")
    
    # Rasterize each timestamp
    transform = from_bounds(minlon, minlat, maxlon, maxlat, width, height)
    
    # Store rasters for each timestamp
    time_rasters = {}
    
    for time_idx, (timestamp, contours_at_time) in enumerate(sorted(contours_by_time.items())):
        print(f"  Rasterizing timestamp {time_idx + 1}/{len(contours_by_time)}: {timestamp}")
        
        # Initialize raster for this timestamp
        time_raster = np.zeros((height, width), dtype=np.float32)
        
        # Rasterize all contours at this timestamp and sum them
        for contour in contours_at_time:
            try:
                coords = contour['coords']
                if len(coords) < 3:
                    continue
                
                poly = Polygon(coords)
                conc = contour['concentration']
                
                # Rasterize polygon
                shapes = [(poly, conc)]
                raster = rasterize(shapes, out_shape=(height, width), transform=transform, default_value=0)
                
                # Add to time raster (sum all contours at this time)
                time_raster += raster
            except Exception as e:
                pass
        
        time_rasters[timestamp] = time_raster
    
    # Now aggregate across time
    print(f"\nAggregating {len(time_rasters)} timesteps...")
    
    conc_sum = np.zeros((height, width), dtype=np.float32)
    conc_max = np.zeros((height, width), dtype=np.float32)
    conc_min = np.full((height, width), np.inf, dtype=np.float32)
    frequency = np.zeros((height, width), dtype=np.int32)
    
    for timestamp, time_raster in time_rasters.items():
        # Where this timestamp has values
        mask = time_raster > 0
        
        # Sum across time
        conc_sum[mask] += time_raster[mask]
        conc_max[mask] = np.maximum(conc_max[mask], time_raster[mask])
        conc_min[mask] = np.minimum(conc_min[mask], time_raster[mask])
        frequency[mask] += 1
    
    # Calculate statistics
    valid = frequency > 0
    avg_conc = np.zeros_like(conc_sum)
    avg_conc[valid] = conc_sum[valid] / frequency[valid]
    
    hazard_quotient = np.zeros_like(avg_conc)
    hazard_quotient[valid] = avg_conc[valid] / STANDARD
    
    # Save as GeoTIFF files
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # WGS 1984 projection
    wgs84 = CRS.from_string('+proj=latlong +datum=WGS84 +no_defs')
    
    # Average concentration
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'avg_concentration.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=avg_conc.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(avg_conc, 1)
    
    # Max concentration
    conc_max[~valid] = 0
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'max_concentration.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=conc_max.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(conc_max, 1)
    
    # Frequency
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'frequency.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=frequency.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(frequency, 1)
    
    # Hazard quotient
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'hazard_quotient.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=hazard_quotient.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(hazard_quotient, 1)
    
    print(f"\nRasters saved to {OUTPUT_DIR}/")
    print(f"Files created:")
    print(f"  - avg_concentration.tif")
    print(f"  - max_concentration.tif")
    print(f"  - frequency.tif")
    print(f"  - hazard_quotient.tif")
    
    # Create CSV with grid cell statistics
    print(f"\nCreating attribute table CSV...")
    csv_path = os.path.join(OUTPUT_DIR, 'grid_attributes.csv')
    
    with open(csv_path, 'w', newline='') as f:
        f.write('row,col,latitude,longitude,avg_concentration,max_concentration,frequency,hazard_quotient\n')
        
        for i in range(height):
            for j in range(width):
                if frequency[i, j] > 0:
                    # Calculate lat/lon from grid position
                    lat = maxlat - (i + 0.5) * grid_res
                    lon = minlon + (j + 0.5) * grid_res
                    
                    f.write(f'{i},{j},{lat:.6f},{lon:.6f},{avg_conc[i, j]:.6f},{conc_max[i, j]:.6f},{int(frequency[i, j])},{hazard_quotient[i, j]:.6f}\n')
    
    print(f"Attribute table saved to {csv_path}")

def main():
    kmz_files = list(Path(INPUT_DIR).glob('*.kmz'))
    
    if not kmz_files:
        print(f"No KMZ files found in {INPUT_DIR}")
        return
    
    print(f"Processing {len(kmz_files)} KMZ files...\n")
    
    all_contours = []
    
    for kmz_path in kmz_files:
        print(f"Extracting contours from: {kmz_path.name}")
        contours = extract_contour_data(str(kmz_path))
        print(f"  Found {len(contours)} contours")
        if contours:
            concs = [c['concentration'] for c in contours]
            print(f"  Concentration range: {min(concs):.3g} to {max(concs):.3g}")
        all_contours.extend(contours)
    
    print(f"\nTotal contours extracted: {len(all_contours)}")
    
    print("\nCreating raster stack...")
    create_raster_stack(all_contours)

if __name__ == "__main__":
    main()

Processing 12 KMZ files...

Extracting contours from: jul24_los_cumpas_pm10.kmz
  Found 74 contours
  Concentration range: 1e-06 to 0.001
Extracting contours from: jul24_san_juan_pm10.kmz
  Found 71 contours
  Concentration range: 1e-07 to 0.0001
Extracting contours from: jul24_san_mig_pm10.kmz
  Found 79 contours
  Concentration range: 1e-06 to 0.001
Extracting contours from: jul24_sn_pm10.kmz
  Found 58 contours
  Concentration range: 1e-05 to 0.01
Extracting contours from: sep5_los_cumpas_pm10.kmz
  Found 77 contours
  Concentration range: 1e-09 to 0.001
Extracting contours from: sep5_san_juan_pm10.kmz
  Found 84 contours
  Concentration range: 1e-11 to 1e-05
Extracting contours from: sep5_san_mig_pm10.kmz
  Found 69 contours
  Concentration range: 1e-06 to 0.001
Extracting contours from: sep5_sn_pm10.kmz
  Found 81 contours
  Concentration range: 1e-09 to 0.001
Extracting contours from: sep8_los_cumpas_pm10.kmz
  Found 73 contours
  Concentration range: 1e-06 to 0.001
Extracting co

In [3]:
#PM2.5 aggregation
import zipfile
import xml.etree.ElementTree as ET
import os
from pathlib import Path
import shutil
from collections import defaultdict
import numpy as np
from scipy import ndimage
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from shapely.geometry import Polygon
from rasterio.crs import CRS

#Set directories
INPUT_DIR = "./kmz_files_pm25"
OUTPUT_DIR = "./hotspot_rasters_pm25_v2"
NS = '{http://www.opengis.net/kml/2.2}'
STANDARD = 0.000015  # Standard for hazard quotient calculation

def extract_contour_data(kmz_path):
    """Extract contour level data from a KMZ file"""
    contours = []
    source_file = Path(kmz_path).stem  # Get filename without extension
    
    try:
        extract_dir = "./temp_kml_contour"
        with zipfile.ZipFile(kmz_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        kml_file = None
        for file in os.listdir(extract_dir):
            if file.endswith('.kml'):
                kml_file = os.path.join(extract_dir, file)
                break
        
        if kml_file is None:
            return contours
        
        tree = ET.parse(kml_file)
        root = tree.getroot()
        
        doc = root.find(f'{NS}Document')
        if doc is None:
            return contours
        
        for folder in doc.findall(f'{NS}Folder'):
            folder_name = folder.find(f'{NS}name')
            if folder_name is None or folder_name.text is None:
                continue
            
            fname = folder_name.text.strip()
            
            if any(skip in fname for skip in ['HYSPLIT', 'NOAA', 'EPA', 'Soure']):
                continue
            
            time_str = fname.replace('<pre>', '').replace('</pre>', '').strip()
            
            for placemark in folder.findall(f'{NS}Placemark'):
                pm_name = placemark.find(f'{NS}name')
                if pm_name is None or pm_name.text is None:
                    continue
                
                name_text = pm_name.text.strip()
                
                if 'Contour Level' in name_text:
                    try:
                        parts = name_text.split(':')
                        if len(parts) > 1:
                            conc_str = parts[1].strip().split()[0]
                            concentration = float(conc_str)
                            
                            coords = None
                            
                            multi_geom = placemark.find(f'{NS}MultiGeometry')
                            if multi_geom is not None:
                                polygons = multi_geom.findall(f'{NS}Polygon')
                                if polygons:
                                    for polygon in polygons:
                                        outer_ring = polygon.find(f'{NS}outerBoundaryIs/{NS}LinearRing/{NS}coordinates')
                                        if outer_ring is not None and outer_ring.text:
                                            coords_text = outer_ring.text.strip()
                                            coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                                            if coords:
                                                break
                                
                                if coords is None:
                                    linestrings = multi_geom.findall(f'{NS}LineString')
                                    if linestrings:
                                        for linestring in linestrings:
                                            line_coords = linestring.find(f'{NS}coordinates')
                                            if line_coords is not None and line_coords.text:
                                                coords_text = line_coords.text.strip()
                                                coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                                                if coords:
                                                    break
                            
                            if coords is None:
                                polygon = placemark.find(f'{NS}Polygon')
                                if polygon is not None:
                                    outer_ring = polygon.find(f'{NS}outerBoundaryIs/{NS}LinearRing/{NS}coordinates')
                                    if outer_ring is not None and outer_ring.text:
                                        coords_text = outer_ring.text.strip()
                                        coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                            
                            if coords is None:
                                linestring = placemark.find(f'{NS}LineString')
                                if linestring is not None:
                                    line_coords = linestring.find(f'{NS}coordinates')
                                    if line_coords is not None and line_coords.text:
                                        coords_text = line_coords.text.strip()
                                        coords = [tuple(map(float, c.split(',')[:2])) for c in coords_text.split() if c]
                            
                            if coords and len(coords) > 0:
                                contours.append({
                                    'time': time_str,
                                    'concentration': concentration,
                                    'coords': coords,
                                    'source': source_file
                                })
                    except (ValueError, IndexError):
                        pass
        
        shutil.rmtree(extract_dir)
        
    except Exception as e:
        print(f"Error processing {kmz_path}: {e}")
    
    return contours

def create_raster_stack(all_contours):
    """Create raster stack from contours, aggregate by timestamp, then calculate statistics"""
    
    if not all_contours:
        print("No contours found!")
        return
    
    # Determine bounds
    all_coords = [c for contour in all_contours for c in contour['coords']]
    lons = [c[0] for c in all_coords]
    lats = [c[1] for c in all_coords]
    
    minlon, maxlon = min(lons), max(lons)
    minlat, maxlat = min(lats), max(lats)
    
    # Add buffer
    buffer = 0.05
    minlon -= buffer
    maxlon += buffer
    minlat -= buffer
    maxlat += buffer
    
    # Create grid (0.01 degree resolution)
    grid_res = 0.01
    width = int((maxlon - minlon) / grid_res)
    height = int((maxlat - minlat) / grid_res)
    
    print(f"Creating raster grid: {width} x {height}")
    
    # Group contours by timestamp AND source location
    contours_by_time_and_source = defaultdict(lambda: defaultdict(list))
    for contour in all_contours:
        time = contour['time']
        source = contour['source']
        contours_by_time_and_source[time][source].append(contour)
    
    print(f"Found {len(contours_by_time_and_source)} unique timestamps")
    
    # Rasterize each timestamp
    transform = from_bounds(minlon, minlat, maxlon, maxlat, width, height)
    
    # Store rasters for each timestamp
    time_rasters = {}
    
    for time_idx, (timestamp, sources_dict) in enumerate(sorted(contours_by_time_and_source.items())):
        print(f"  Rasterizing timestamp {time_idx + 1}/{len(contours_by_time_and_source)}: {timestamp}")
        
        # Initialize raster for this timestamp
        time_raster = np.zeros((height, width), dtype=np.float32)
        
        # For each source location
        for source, contours_at_source in sources_dict.items():
            # Initialize raster for this source
            source_raster = np.zeros((height, width), dtype=np.float32)
            
            # Rasterize all contours for this source and take MAXIMUM
            for contour in contours_at_source:
                try:
                    coords = contour['coords']
                    if len(coords) < 3:
                        continue
                    
                    poly = Polygon(coords)
                    conc = contour['concentration']
                    
                    # Rasterize polygon
                    shapes = [(poly, conc)]
                    raster = rasterize(shapes, out_shape=(height, width), transform=transform, default_value=0)
                    
                    # Take maximum within this source (for overlapping contours)
                    source_raster = np.maximum(source_raster, raster)
                except Exception as e:
                    pass
            
            # Add source raster to time raster (different locations get added)
            time_raster += source_raster
        
        time_rasters[timestamp] = time_raster
    
    # Now aggregate across time
    print(f"\nAggregating {len(time_rasters)} timesteps...")
    
    conc_sum = np.zeros((height, width), dtype=np.float32)
    conc_max = np.zeros((height, width), dtype=np.float32)
    conc_min = np.full((height, width), np.inf, dtype=np.float32)
    frequency = np.zeros((height, width), dtype=np.int32)
    
    # Store all concentration values for each cell to calculate percentile
    conc_values = defaultdict(list)
    
    for timestamp, time_raster in time_rasters.items():
        # Where this timestamp has values
        mask = time_raster > 0
        
        # Sum across time
        conc_sum[mask] += time_raster[mask]
        conc_max[mask] = np.maximum(conc_max[mask], time_raster[mask])
        conc_min[mask] = np.minimum(conc_min[mask], time_raster[mask])
        frequency[mask] += 1
        
        # Store individual values for percentile calculation
        indices = np.argwhere(mask)
        for i, j in indices:
            conc_values[(i, j)].append(time_raster[i, j])
    
    # Calculate statistics
    valid = frequency > 0
    avg_conc = np.zeros_like(conc_sum)
    avg_conc[valid] = conc_sum[valid] / frequency[valid]
    
    # Calculate 95th percentile for each cell
    percentile_95 = np.zeros_like(conc_sum)
    for (i, j), values in conc_values.items():
        if values:
            percentile_95[i, j] = np.percentile(values, 95)
    
    # Calculate hazard quotient using 95th percentile
    hazard_quotient = np.zeros_like(percentile_95)
    hazard_quotient[valid] = percentile_95[valid] / STANDARD
    
    # Save as GeoTIFF files
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # WGS 1984 projection
    wgs84 = CRS.from_string('+proj=latlong +datum=WGS84 +no_defs')
    
    # Mean concentration
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'mean_concentration.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=avg_conc.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(avg_conc, 1)
    
    # Max concentration
    conc_max[~valid] = 0
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'max_concentration.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=conc_max.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(conc_max, 1)
    
    # Frequency (number of timesteps with data)
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'frequency_timesteps.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=frequency.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(frequency, 1)
    
    # 95th percentile concentration
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'percentile_95_concentration.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=percentile_95.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(percentile_95, 1)
    
    # Hazard quotient
    with rasterio.open(
        os.path.join(OUTPUT_DIR, 'hazard_quotient.tif'),
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=hazard_quotient.dtype,
        transform=transform,
        crs=wgs84
    ) as dst:
        dst.write(hazard_quotient, 1)
    
    print(f"\nRasters saved to {OUTPUT_DIR}/")
    print(f"Files created:")
    print(f"  - mean_concentration.tif")
    print(f"  - max_concentration.tif")
    print(f"  - frequency_timesteps.tif")
    print(f"  - percentile_95_concentration.tif")
    print(f"  - hazard_quotient.tif")
    
    # Create CSV with grid cell statistics
    print(f"\nCreating attribute table CSV...")
    csv_path = os.path.join(OUTPUT_DIR, 'grid_attributes.csv')
    
    with open(csv_path, 'w', newline='') as f:
        f.write('row,col,latitude,longitude,mean_concentration,max_concentration,frequency_timesteps,percentile_95_concentration,hazard_quotient\n')
        
        for i in range(height):
            for j in range(width):
                if frequency[i, j] > 0:
                    # Calculate lat/lon from grid position
                    lat = maxlat - (i + 0.5) * grid_res
                    lon = minlon + (j + 0.5) * grid_res
                    
                    f.write(f'{i},{j},{lat:.6f},{lon:.6f},{avg_conc[i, j]:.6f},{conc_max[i, j]:.6f},{int(frequency[i, j])},{percentile_95[i, j]:.6f},{hazard_quotient[i, j]:.6f}\n')
    
    print(f"Attribute table saved to {csv_path}")

def main():
    kmz_files = list(Path(INPUT_DIR).glob('*.kmz'))
    
    if not kmz_files:
        print(f"No KMZ files found in {INPUT_DIR}")
        return
    
    print(f"Processing {len(kmz_files)} KMZ files...\n")
    
    all_contours = []
    
    for kmz_path in kmz_files:
        print(f"Extracting contours from: {kmz_path.name}")
        contours = extract_contour_data(str(kmz_path))
        print(f"  Found {len(contours)} contours")
        if contours:
            concs = [c['concentration'] for c in contours]
            print(f"  Concentration range: {min(concs):.3g} to {max(concs):.3g}")
        all_contours.extend(contours)
    
    print(f"\nTotal contours extracted: {len(all_contours)}")
    
    # Debug: show unique time strings
    unique_times = set(c['time'] for c in all_contours)
    print(f"\nUnique timestamps found: {len(unique_times)}")
    for t in sorted(unique_times)[:5]:
        print(f"  Example: {t}")
    
    print("\nCreating raster stack...")
    create_raster_stack(all_contours)

if __name__ == "__main__":
    main()

Processing 12 KMZ files...

Extracting contours from: jul24_los_cumpas_pm25.kmz
  Found 56 contours
  Concentration range: 1e-06 to 0.001
Extracting contours from: jul24_san_juan_pm25.kmz
  Found 69 contours
  Concentration range: 1e-08 to 1e-05
Extracting contours from: jul24_san_mig_pm25.kmz
  Found 57 contours
  Concentration range: 1e-06 to 0.001
Extracting contours from: jul24_sn_pm25.kmz
  Found 64 contours
  Concentration range: 1e-06 to 0.001
Extracting contours from: sep5_los_cumpas_pm25.kmz
  Found 85 contours
  Concentration range: 1e-08 to 1e-05
Extracting contours from: sep5_san_juan_pm25.kmz
  Found 82 contours
  Concentration range: 1e-09 to 1e-06
Extracting contours from: sep5_san_mig_pm25.kmz
  Found 85 contours
  Concentration range: 1e-08 to 1e-05
Extracting contours from: sep5_sn_pm25.kmz
  Found 72 contours
  Concentration range: 1e-07 to 0.0001
Extracting contours from: sep8_los_cumpas_pm25.kmz
  Found 68 contours
  Concentration range: 1e-07 to 0.0001
Extracting 